# AGA — Controlled Refinement (ImageNet-1K)

## 1. Environment and dependencies

In [ ]:

import os, sys, gc, math, random, json, time
from pathlib import Path

import numpy as np
import torch

assert torch.cuda.is_available(), "Kaggle Notebook에서 GPU accelerator를 켜주세요."

WORK = Path("/kaggle/working")
DINO_REPO = WORK / "dinov2"
OUTPUT_ROOT = WORK / "aga_server_run"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUTPUT_ROOT)


In [ ]:

!pip install -q -U datasets huggingface_hub

if not DINO_REPO.exists():
    !git clone -q https://github.com/facebookresearch/dinov2.git /kaggle/working/dinov2

if str(DINO_REPO) not in sys.path:
    sys.path.insert(0, str(DINO_REPO))

import dinov2
print("dinov2 import OK:", dinov2.__version__)


In [ ]:

from kaggle_secrets import UserSecretsClient
from huggingface_hub import whoami, HfApi

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Kaggle Secret 'HF_TOKEN'을 읽지 못했습니다. "
        "Add-ons → Secrets에서 등록하고 notebook access를 허용하세요."
    ) from e

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN Secret이 비어 있습니다.")

os.environ["HF_TOKEN"] = HF_TOKEN

hf_user = whoami(token=HF_TOKEN)
dataset_info = HfApi(token=HF_TOKEN).dataset_info(
    "ILSVRC/imagenet-1k",
    token=HF_TOKEN,
)

print("Hugging Face auth OK:", hf_user["name"])
print("Dataset access OK:", dataset_info.id)


## 2. Experiment configuration

In [ ]:

SEED = 42
HF_DATASET = "ILSVRC/imagenet-1k"

# Data
BATCH_SIZE = 80
LOCAL_CROPS = 4
GLOBAL_SIZE = 224
LOCAL_SIZE = 98
GRAM_SIZE = 448
SHUFFLE_BUFFER = 2048
NUM_WORKERS = 0  # HF IterableDataset 중복 worker 방지를 위해 유지

# Heads
DINO_OUT_DIM = 8192
IBOT_OUT_DIM = 8192
HEAD_HIDDEN_DIM = 2048
HEAD_BOTTLENECK_DIM = 256
HEAD_NLAYERS = 3

# Loss weights
DINO_WEIGHT = 1.0
IBOT_WEIGHT = 1.0
KOLEO_WEIGHT = 0.1
RELATION_WEIGHT = 1.0

AGA_B = 0.4
AGA_ALPHA = 8.0
HIGHAM_MAX_ITER = 0
HIGHAM_TOL = 1e-4

# Training
CALIB_STEPS = 20
REFINE_STEPS = 1500

HEAD_CALIB_LR = 1e-4
BACKBONE_LR = 1e-5
HEAD_REFINE_LR = 1e-4
WEIGHT_DECAY = 0.04
GRAD_CLIP = 3.0

TEACHER_TEMP = 0.07
EMA_MOMENTUM = 0.999
USE_AMP = True

# unattended run controls
#RUN_MODES = ["none", "gram", "aga"]
RUN_MODES = ["aga"]
PRINT_EVERY = 5
SAVE_EVERY = 500

CONFIG = {
    k: v for k, v in globals().copy().items()
    if k in {
        "SEED", "BATCH_SIZE", "LOCAL_CROPS", "GLOBAL_SIZE", "LOCAL_SIZE",
        "GRAM_SIZE", "SHUFFLE_BUFFER", "NUM_WORKERS", "DINO_OUT_DIM",
        "IBOT_OUT_DIM", "HEAD_HIDDEN_DIM", "HEAD_BOTTLENECK_DIM",
        "HEAD_NLAYERS", "DINO_WEIGHT", "IBOT_WEIGHT", "KOLEO_WEIGHT",
        "RELATION_WEIGHT", "AGA_B", "AGA_ALPHA", "HIGHAM_MAX_ITER",
        "HIGHAM_TOL", "CALIB_STEPS", "REFINE_STEPS", "HEAD_CALIB_LR",
        "BACKBONE_LR", "HEAD_REFINE_LR", "WEIGHT_DECAY", "GRAD_CLIP",
        "TEACHER_TEMP", "EMA_MOMENTUM", "USE_AMP", "RUN_MODES",
        "PRINT_EVERY", "SAVE_EVERY"
    }
}

with open(OUTPUT_ROOT / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2))


## 3. Utilities

In [ ]:

import psutil
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

from functools import partial
from torch.utils.data import IterableDataset, DataLoader
from torchvision import transforms
from datasets import load_dataset

from dinov2.data import MaskingGenerator, collate_data_and_cast
from dinov2.data.transforms import GaussianBlur, make_normalize_transform
from dinov2.hub.backbones import dinov2_vits14
from dinov2.layers import DINOHead
from dinov2.loss import DINOLoss, iBOTPatchLoss, KoLeoLoss


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def memory_status(prefix=""):
    p = psutil.Process(os.getpid())
    cpu = p.memory_info().rss / 1024**3
    gpu_alloc = torch.cuda.memory_allocated() / 1024**3
    gpu_res = torch.cuda.memory_reserved() / 1024**3
    print(
        f"{prefix} CPU RSS={cpu:.2f} GB | "
        f"GPU allocated={gpu_alloc:.2f} GB | "
        f"GPU reserved={gpu_res:.2f} GB"
    )


def cleanup_cuda():
    gc.collect()
    torch.cuda.empty_cache()


## 4. ImageNet streaming and crops

In [ ]:

class DINOAugmentation:
    """
    Same global/local augmentation as the previous notebook.
    For Gram/AGA only, also returns clean 448x448 crops sharing
    the exact global crop geometry.
    """
    def __init__(
        self,
        include_gram=False,
        global_scale=(0.32, 1.0),
        local_scale=(0.05, 0.32),
        local_crops_number=4,
        global_size=224,
        local_size=98,
        gram_size=448,
    ):
        self.include_gram = include_gram
        self.local_crops_number = local_crops_number

        self.global_geometry = transforms.Compose([
            transforms.RandomResizedCrop(
                gram_size,
                scale=global_scale,
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(p=0.5),
        ])

        self.resize_global = transforms.Resize(
            global_size,
            interpolation=transforms.InterpolationMode.BICUBIC,
        )

        self.local_geometry = transforms.Compose([
            transforms.RandomResizedCrop(
                local_size,
                scale=local_scale,
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(p=0.5),
        ])

        color_jitter = transforms.Compose([
            transforms.RandomApply(
                [transforms.ColorJitter(
                    brightness=0.4,
                    contrast=0.4,
                    saturation=0.2,
                    hue=0.1,
                )],
                p=0.8,
            ),
            transforms.RandomGrayscale(p=0.2),
        ])

        normalize = transforms.Compose([
            transforms.ToTensor(),
            make_normalize_transform(),
        ])
        self.normalize = normalize

        self.global1 = transforms.Compose([
            color_jitter,
            GaussianBlur(p=1.0),
            normalize,
        ])

        self.global2 = transforms.Compose([
            color_jitter,
            GaussianBlur(p=0.1),
            transforms.RandomSolarize(threshold=128, p=0.2),
            normalize,
        ])

        self.local = transforms.Compose([
            color_jitter,
            GaussianBlur(p=0.5),
            normalize,
        ])

    def __call__(self, image):
        image = image.convert("RGB")

        # Always create the same global geometry.
        base1 = self.global_geometry(image)
        base2 = self.global_geometry(image)

        g1 = self.global1(self.resize_global(base1))
        g2 = self.global2(self.resize_global(base2))

        local_crops = [
            self.local(self.local_geometry(image))
            for _ in range(self.local_crops_number)
        ]

        out = {
            "global_crops": [g1, g2],
            "global_crops_teacher": [g1, g2],
            "local_crops": local_crops,
            "offsets": (),
        }

        # Deterministic extra work only for relation modes.
        if self.include_gram:
            out["gram_teacher_crops"] = [
                self.normalize(base1),
                self.normalize(base2),
            ]

        return out


class HFImageNetStream(IterableDataset):
    def __init__(self, hf_stream, transform):
        super().__init__()
        self.hf_stream = hf_stream
        self.transform = transform

    def __iter__(self):
        for sample in self.hf_stream:
            try:
                image = sample["image"].convert("RGB")
                yield self.transform(image), ()
            except Exception:
                continue


def collate_optional_gram(
    samples_list,
    mask_ratio_tuple,
    mask_probability,
    dtype,
    n_tokens,
    mask_generator,
):
    batch = collate_data_and_cast(
        samples_list,
        mask_ratio_tuple=mask_ratio_tuple,
        mask_probability=mask_probability,
        dtype=dtype,
        n_tokens=n_tokens,
        mask_generator=mask_generator,
    )

    first = samples_list[0][0]
    if "gram_teacher_crops" in first:
        n_gram = len(first["gram_teacher_crops"])
        batch["collated_gram_teacher_crops"] = torch.stack([
            sample[0]["gram_teacher_crops"][i]
            for i in range(n_gram)
            for sample in samples_list
        ])

    return batch


def build_stream_loader(mode, seed=SEED):
    include_gram = mode in {"gram", "aga"}

    stream = load_dataset(
        HF_DATASET,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    ).shuffle(
        seed=seed,
        buffer_size=SHUFFLE_BUFFER,
    )

    transform = DINOAugmentation(
        include_gram=include_gram,
        local_crops_number=LOCAL_CROPS,
        global_size=GLOBAL_SIZE,
        local_size=LOCAL_SIZE,
        gram_size=GRAM_SIZE,
    )

    dataset = HFImageNetStream(stream, transform)

    n_tokens = (GLOBAL_SIZE // 14) ** 2
    mask_generator = MaskingGenerator(
        input_size=(GLOBAL_SIZE // 14, GLOBAL_SIZE // 14),
        max_num_patches=int(0.5 * n_tokens),
    )

    collate_fn = partial(
        collate_optional_gram,
        mask_ratio_tuple=(0.1, 0.5),
        mask_probability=0.5,
        dtype=torch.float32,
        n_tokens=n_tokens,
        mask_generator=mask_generator,
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True,
        drop_last=True,
    )


## 5. Gram and AGA losses

In [ ]:

class OriginalGramLoss(nn.Module):
    def forward(self, student_feats, teacher_feats):
        with torch.autocast(device_type="cuda", enabled=False):
            s = F.normalize(student_feats.float(), dim=-1)
            t = F.normalize(teacher_feats.float(), dim=-1)

            Gs = s @ s.transpose(-1, -2)

            with torch.no_grad():
                Gt = t @ t.transpose(-1, -2)

            return F.mse_loss(Gs, Gt), {}


class AmplifiedGramLoss(nn.Module):
    def __init__(
        self,
        b=0.5,
        alpha=4.0,
        max_iter=50,
        tol=1e-4,
        eps=1e-8,
    ):
        super().__init__()
        self.b = b
        self.alpha = alpha
        self.max_iter = max_iter
        self.tol = tol
        self.eps = eps
        self.last_iters = 0

    def nonlin_proj(self, G):
        denom = torch.tanh(
            torch.as_tensor(
                self.alpha * (1.0 - self.b),
                device=G.device,
                dtype=G.dtype,
            )
        ).clamp_min(self.eps)

        amp = (
            self.b
            + (1.0 - self.b)
            * torch.tanh(self.alpha * (G - self.b))
            / denom
        )

        return torch.where(G > self.b, amp, G)

    @staticmethod
    def psd_proj(R):
        R = 0.5 * (R + R.transpose(-1, -2))
        eigvals, eigvecs = torch.linalg.eigh(R)
        eigvals = eigvals.clamp_min(0.0)
        X = (eigvecs * eigvals.unsqueeze(-2)) @ eigvecs.transpose(-1, -2)
        return 0.5 * (X + X.transpose(-1, -2))

    @staticmethod
    def unitdiag_proj(X):
        Y = X.clone()
        Y.diagonal(dim1=-2, dim2=-1).fill_(1.0)
        return Y

    @torch.no_grad()
    def higham(self, A):
        Y = A.clone()
        S = torch.zeros_like(A)
    
        for i in range(self.max_iter):
            old = Y
    
            R = Y - S
            X = self.psd_proj(R)
            S = X - R
            Y = self.unitdiag_proj(X)
    
            rel = (
                torch.linalg.matrix_norm(
                    Y - old,
                    ord="fro",
                    dim=(-2, -1),
                )
                /
                (
                    torch.linalg.matrix_norm(
                        old,
                        ord="fro",
                        dim=(-2, -1),
                    )
                    + self.eps
                )
            )
    
            self.last_conv_error = rel.max().item()
    
            if rel.max() < self.tol:
                self.last_iters = i + 1
                break
    
        else:
            self.last_iters = self.max_iter
    
        return Y

    @torch.no_grad()
    def stats(self, G, G_amp, G_hat):
        tri = torch.triu(
            torch.ones_like(G, dtype=torch.bool),
            diagonal=1,
        )
        strong = tri & (G > self.b)
        weak = tri & (G <= self.b)

        def mean_or_nan(x, mask):
            if mask.any():
                return x[mask].mean()
            return torch.tensor(float("nan"), device=x.device)

        d_amp = mean_or_nan(G_amp - G, strong)
        d_hat = mean_or_nan(G_hat - G, strong)
        d_weak = mean_or_nan((G_hat - G).abs(), weak)

        if torch.isfinite(d_amp) and d_amp.abs() > self.eps:
            retention = d_hat / d_amp
        else:
            retention = torch.tensor(float("nan"), device=G.device)

        return {
            "d_amp": float(d_amp),
            "d_hat": float(d_hat),
            "retention": float(retention),
            "d_weak": float(d_weak),
            "strong_ratio": float(strong.sum() / tri.sum().clamp_min(1)),
            "higham_iters": self.last_iters,
        }

    def forward(self, student_feats, teacher_feats):
        with torch.autocast(device_type="cuda", enabled=False):
            s = F.normalize(student_feats.float(), dim=-1)
            Gs = s @ s.transpose(-1, -2)

            with torch.no_grad():
                t = F.normalize(teacher_feats.float(), dim=-1)
                G = t @ t.transpose(-1, -2)
                G_amp = self.nonlin_proj(G)
                G_hat = self.higham(G_amp)
                stats = self.stats(G, G_amp, G_hat)

            return F.mse_loss(Gs, G_hat), stats


## 6. DINOv2 student, EMA teacher and frozen Gram teacher

In [ ]:

def resize_patch_tokens(tokens, src_size, dst_size, patch_size=14):
    src_n = src_size // patch_size
    dst_n = dst_size // patch_size

    assert tokens.shape[1] == src_n * src_n

    x = tokens.transpose(1, 2).reshape(
        tokens.shape[0],
        tokens.shape[2],
        src_n,
        src_n,
    )

    x = F.interpolate(
        x,
        size=(dst_n, dst_n),
        mode="bicubic",
        align_corners=False,
        antialias=False,
    )

    return x.flatten(2).transpose(1, 2)


class DINOv2Refiner(nn.Module):
    def __init__(self, relation_mode="none"):
        super().__init__()
        assert relation_mode in {"none", "gram", "aga"}

        self.relation_mode = relation_mode

        self.student_backbone = dinov2_vits14(pretrained=True)

        self.teacher_backbone = dinov2_vits14(pretrained=False)
        self.teacher_backbone.load_state_dict(
            self.student_backbone.state_dict(),
            strict=True,
        )

        # Only relation runs need a third ViT.
        if relation_mode in {"gram", "aga"}:
            self.gram_teacher = dinov2_vits14(pretrained=False)
            self.gram_teacher.load_state_dict(
                self.student_backbone.state_dict(),
                strict=True,
            )
        else:
            self.gram_teacher = None

        self.student_dino_head = DINOHead(
            in_dim=384,
            out_dim=DINO_OUT_DIM,
            hidden_dim=HEAD_HIDDEN_DIM,
            bottleneck_dim=HEAD_BOTTLENECK_DIM,
            nlayers=HEAD_NLAYERS,
        )
        self.student_ibot_head = DINOHead(
            in_dim=384,
            out_dim=IBOT_OUT_DIM,
            hidden_dim=HEAD_HIDDEN_DIM,
            bottleneck_dim=HEAD_BOTTLENECK_DIM,
            nlayers=HEAD_NLAYERS,
        )
        self.teacher_dino_head = DINOHead(
            in_dim=384,
            out_dim=DINO_OUT_DIM,
            hidden_dim=HEAD_HIDDEN_DIM,
            bottleneck_dim=HEAD_BOTTLENECK_DIM,
            nlayers=HEAD_NLAYERS,
        )
        self.teacher_ibot_head = DINOHead(
            in_dim=384,
            out_dim=IBOT_OUT_DIM,
            hidden_dim=HEAD_HIDDEN_DIM,
            bottleneck_dim=HEAD_BOTTLENECK_DIM,
            nlayers=HEAD_NLAYERS,
        )

        self.teacher_dino_head.load_state_dict(
            self.student_dino_head.state_dict(),
            strict=True,
        )
        self.teacher_ibot_head.load_state_dict(
            self.student_ibot_head.state_dict(),
            strict=True,
        )

        self.dino_loss = DINOLoss(DINO_OUT_DIM)
        self.ibot_loss = iBOTPatchLoss(IBOT_OUT_DIM)
        self.koleo_loss = KoLeoLoss()

        if relation_mode == "gram":
            self.relation_loss = OriginalGramLoss()
        elif relation_mode == "aga":
            self.relation_loss = AmplifiedGramLoss(
                b=AGA_B,
                alpha=AGA_ALPHA,
                max_iter=HIGHAM_MAX_ITER,
                tol=HIGHAM_TOL,
            )
        else:
            self.relation_loss = None

        self.freeze_teachers()

    def freeze_teachers(self):
        modules = [
            self.teacher_backbone,
            self.teacher_dino_head,
            self.teacher_ibot_head,
        ]
        if self.gram_teacher is not None:
            modules.append(self.gram_teacher)

        for module in modules:
            module.requires_grad_(False)
            module.eval()

    def train(self, mode=True):
        super().train(mode)
        self.teacher_backbone.eval()
        self.teacher_dino_head.eval()
        self.teacher_ibot_head.eval()
        if self.gram_teacher is not None:
            self.gram_teacher.eval()
        return self

    def set_backbone_trainable(self, trainable):
        self.student_backbone.requires_grad_(trainable)

    @torch.no_grad()
    def reset_gram_teacher_from_student(self):
        if self.gram_teacher is not None:
            self.gram_teacher.load_state_dict(
                self.student_backbone.state_dict(),
                strict=True,
            )
            self.gram_teacher.requires_grad_(False)
            self.gram_teacher.eval()

    @torch.no_grad()
    def update_teacher(self, m=EMA_MOMENTUM):
        pairs = [
            (self.student_backbone, self.teacher_backbone),
            (self.student_dino_head, self.teacher_dino_head),
            (self.student_ibot_head, self.teacher_ibot_head),
        ]

        for student, teacher in pairs:
            for ps, pt in zip(student.parameters(), teacher.parameters()):
                pt.mul_(m).add_(ps, alpha=1.0 - m)

    def forward_losses(
        self,
        batch,
        teacher_temp=TEACHER_TEMP,
        relation_weight=RELATION_WEIGHT,
        calibration=False,
    ):
        device = next(self.student_backbone.parameters()).device

        global_crops = batch["collated_global_crops"].to(
            device, non_blocking=True
        )
        local_crops = batch["collated_local_crops"].to(
            device, non_blocking=True
        )
        masks = batch["collated_masks"].to(
            device, non_blocking=True
        )
        mask_indices = batch["mask_indices_list"].to(
            device, non_blocking=True
        )
        masks_weight = batch["masks_weight"].to(
            device, non_blocking=True
        )

        n_global = 2
        n_local = LOCAL_CROPS
        n_masked = mask_indices.shape[0]

        with torch.no_grad():
            t = self.teacher_backbone(
                global_crops,
                is_training=True,
            )
            t_cls = t["x_norm_clstoken"]
            t_patch = t["x_norm_patchtokens"]

            c0, c1 = t_cls.chunk(2)
            t_cls_rev = torch.cat([c1, c0], dim=0)

            t_cls_logits = self.teacher_dino_head(t_cls_rev)
            t_cls_probs = self.dino_loss.softmax_center_teacher(
                t_cls_logits,
                teacher_temp=teacher_temp,
            ).view(2, -1, DINO_OUT_DIM)
            self.dino_loss.update_center(t_cls_logits)

            t_masked_patch = torch.index_select(
                t_patch.flatten(0, 1),
                dim=0,
                index=mask_indices,
            )

            t_patch_logits = self.teacher_ibot_head(t_masked_patch)
            t_patch_logits_3d = t_patch_logits.unsqueeze(0)

            t_patch_probs = self.ibot_loss.softmax_center_teacher(
                t_patch_logits_3d,
                teacher_temp=teacher_temp,
            ).squeeze(0)
            self.ibot_loss.update_center(t_patch_logits_3d)

            gt_patch = None
            if (not calibration) and self.gram_teacher is not None:
                gram_crops = batch[
                    "collated_gram_teacher_crops"
                ].to(device, non_blocking=True)

                gt = self.gram_teacher(
                    gram_crops,
                    is_training=True,
                )
                gt_patch = resize_patch_tokens(
                    gt["x_norm_patchtokens"],
                    src_size=GRAM_SIZE,
                    dst_size=GLOBAL_SIZE,
                    patch_size=14,
                )

        s_global = self.student_backbone(
            global_crops,
            masks=masks,
            is_training=True,
        )
        s_local = self.student_backbone(
            local_crops,
            is_training=True,
        )

        s_g_cls = s_global["x_norm_clstoken"]
        s_l_cls = s_local["x_norm_clstoken"]
        s_g_patch = s_global["x_norm_patchtokens"]

        s_g_cls_logits = self.student_dino_head(s_g_cls)
        s_l_cls_logits = self.student_dino_head(s_l_cls)

        s_masked_patch = torch.index_select(
            s_g_patch.flatten(0, 1),
            dim=0,
            index=mask_indices,
        )
        s_masked_patch_logits = self.student_ibot_head(
            s_masked_patch
        )

        n_local_terms = max(n_local * n_global, 1)
        n_global_terms = (n_global - 1) * n_global
        denom = n_local_terms + n_global_terms

        dino_local = (
            self.dino_loss(
                student_output_list=s_l_cls_logits.chunk(n_local),
                teacher_out_softmaxed_centered_list=t_cls_probs,
            )
            / denom
        )

        dino_global = (
            self.dino_loss(
                student_output_list=[s_g_cls_logits],
                teacher_out_softmaxed_centered_list=[
                    t_cls_probs.flatten(0, 1)
                ],
            )
            * 2.0
            / denom
        )

        dino = dino_local + dino_global

        ibot = self.ibot_loss.forward_masked(
            s_masked_patch_logits,
            t_patch_probs,
            student_masks_flat=masks,
            n_masked_patches=n_masked,
            masks_weight=masks_weight,
        )

        koleo = sum(
            self.koleo_loss(x)
            for x in s_g_cls.chunk(2)
        )

        relation_stats = {}

        if calibration:
            relation = dino.new_zeros(())
            total = (
                DINO_WEIGHT * dino
                + IBOT_WEIGHT * ibot
            )
        else:
            if self.relation_loss is not None:
                relation, relation_stats = self.relation_loss(
                    s_g_patch,
                    gt_patch,
                )
            else:
                relation = dino.new_zeros(())

            total = (
                DINO_WEIGHT * dino
                + IBOT_WEIGHT * ibot
                + KOLEO_WEIGHT * koleo
                + relation_weight * relation
            )

        stats = {
            "total": float(total.detach()),
            "dino": float(dino.detach()),
            "ibot": float(ibot.detach()),
            "koleo": float(koleo.detach()),
            "relation": float(relation.detach()),
        }
        stats.update(relation_stats)

        return total, stats


## 7. Optimizers and training loop

In [ ]:

def make_calibration_optimizer(model):
    return torch.optim.AdamW(
        [
            {
                "params": model.student_dino_head.parameters(),
                "lr": HEAD_CALIB_LR,
            },
            {
                "params": model.student_ibot_head.parameters(),
                "lr": HEAD_CALIB_LR,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )


def make_refinement_optimizer(model):
    return torch.optim.AdamW(
        [
            {
                "params": model.student_backbone.parameters(),
                "lr": BACKBONE_LR,
            },
            {
                "params": model.student_dino_head.parameters(),
                "lr": HEAD_REFINE_LR,
            },
            {
                "params": model.student_ibot_head.parameters(),
                "lr": HEAD_REFINE_LR,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )


def save_student_backbone(model, mode, step):
    mode_dir = OUTPUT_ROOT / mode
    mode_dir.mkdir(parents=True, exist_ok=True)

    path = mode_dir / f"student_step_{step:05d}.pt"
    torch.save(
        model.student_backbone.state_dict(),
        path,
    )
    print("checkpoint:", path)


def run_steps(
    model,
    loader,
    optimizer,
    steps,
    mode,
    calibration=False,
    relation_weight=RELATION_WEIGHT,
    print_every=PRINT_EVERY,
    save_every=None,
):
    model.train()
    iterator = iter(loader)

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )
    logs = []

    for step in range(1, steps + 1):
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            loss, stats = model.forward_losses(
                batch,
                relation_weight=relation_weight,
                calibration=calibration,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        trainable_params = [
            p for p in model.parameters()
            if p.requires_grad and p.grad is not None
        ]
        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            GRAD_CLIP,
        )

        scaler.step(optimizer)
        scaler.update()
        model.update_teacher(EMA_MOMENTUM)

        logs.append({"step": step, **stats})

        if step == 1 or step % print_every == 0 or step == steps:
            msg = (
                f"[{mode} {step:04d}/{steps}] "
                f"total={stats['total']:.4f} "
                f"dino={stats['dino']:.4f} "
                f"ibot={stats['ibot']:.4f} "
                f"koleo={stats['koleo']:.4f} "
                f"rel={stats['relation']:.5f}"
            )

            if "retention" in stats:
                msg += (
                    f" | d_amp={stats['d_amp']:.4f}"
                    f" d_hat={stats['d_hat']:.4f}"
                    f" R={stats['retention']:.3f}"
                    f" weak={stats['d_weak']:.4f}"
                    f" strong={stats['strong_ratio']:.3f}"
                    f" H={stats['higham_iters']}"
                )

            print(msg)
            memory_status("  ")

        if (
            (not calibration)
            and save_every is not None
            and step % save_every == 0
        ):
            save_student_backbone(
                model,
                mode,
                step,
            )

    return pd.DataFrame(logs)


## 8. Head calibration

In [ ]:

device = torch.device("cuda")

seed_everything(SEED)

calib_loader = build_stream_loader(
    mode="none",
    seed=SEED,
)

calib_model = DINOv2Refiner(
    relation_mode="none"
).to(device)

calib_model.set_backbone_trainable(False)

calib_optimizer = make_calibration_optimizer(
    calib_model
)

seed_everything(SEED)

memory_status("before calibration:")

calib_logs = run_steps(
    calib_model,
    calib_loader,
    calib_optimizer,
    steps=CALIB_STEPS,
    mode="calibration",
    calibration=True,
    print_every=max(10, PRINT_EVERY),
    save_every=None,
)

CALIB_CKPT = OUTPUT_ROOT / "head_calibrated.pt"

torch.save(
    {
        "model": calib_model.state_dict(),
        "seed": SEED,
        "calib_steps": CALIB_STEPS,
    },
    CALIB_CKPT,
)

calib_logs.to_csv(
    OUTPUT_ROOT / "calibration_logs.csv",
    index=False,
)

# Student backbone is still original pretrained DINOv2-S
# because calibration freezes it. Save as t=0 patch-sim reference.
torch.save(
    calib_model.student_backbone.state_dict(),
    OUTPUT_ROOT / "student_pretrained_reference.pt",
)

print("saved:", CALIB_CKPT)

# IMPORTANT: no calibration objects survive into refinement.
del calib_logs
del calib_optimizer
del calib_model
del calib_loader
cleanup_cuda()

memory_status("after calibration cleanup:")


## 9. Controlled refinement

In [ ]:

def load_calibrated_model(mode):
    model = DINOv2Refiner(
        relation_mode=mode
    ).to(device)

    ckpt = torch.load(
        CALIB_CKPT,
        map_location="cpu",
        weights_only=False,
    )

    if mode == "none":
        model.load_state_dict(
            ckpt["model"],
            strict=True,
        )
    else:
        incompatible = model.load_state_dict(
            ckpt["model"],
            strict=False,
        )

        unexpected = list(incompatible.unexpected_keys)
        missing_non_gram = [
            k for k in incompatible.missing_keys
            if not k.startswith("gram_teacher.")
        ]

        if unexpected or missing_non_gram:
            raise RuntimeError(
                "Unexpected checkpoint mismatch:\n"
                f"unexpected={unexpected}\n"
                f"missing_non_gram={missing_non_gram}"
            )

        model.reset_gram_teacher_from_student()

    model.set_backbone_trainable(True)
    return model


def run_refinement(mode):
    assert mode in {"none", "gram", "aga"}

    print("\n" + "=" * 72)
    print("START:", mode)
    print("=" * 72)

    # Build objects first.
    model = load_calibrated_model(mode)
    loader = build_stream_loader(
        mode=mode,
        seed=SEED,
    )
    optimizer = make_refinement_optimizer(
        model
    )

    # Reset again after mode-specific model construction so that
    # student stochasticity / augmentation / masking start identically.
    seed_everything(SEED)

    memory_status(f"before {mode}:")

    try:
        df = run_steps(
            model,
            loader,
            optimizer,
            steps=REFINE_STEPS,
            mode=mode,
            calibration=False,
            relation_weight=RELATION_WEIGHT,
            print_every=PRINT_EVERY,
            save_every=SAVE_EVERY,
        )

        mode_dir = OUTPUT_ROOT / mode
        mode_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        # Final checkpoint is always written even if REFINE_STEPS
        # is not divisible by SAVE_EVERY.
        final_path = (
            mode_dir
            / f"student_step_{REFINE_STEPS:05d}.pt"
        )
        if not final_path.exists():
            torch.save(
                model.student_backbone.state_dict(),
                final_path,
            )

        df.to_csv(
            mode_dir / "training_log.csv",
            index=False,
        )

        summary_cols = [
            "total",
            "dino",
            "ibot",
            "koleo",
            "relation",
        ]
        summary = {
            "mode": mode,
            **{
                col: float(df[col].mean())
                for col in summary_cols
            },
        }

        for col in [
            "d_amp",
            "d_hat",
            "retention",
            "d_weak",
            "strong_ratio",
            "higham_iters",
        ]:
            if col in df.columns:
                summary[col] = float(
                    df[col].mean()
                )

        with open(
            mode_dir / "summary.json",
            "w",
        ) as f:
            json.dump(
                summary,
                f,
                indent=2,
            )

        print("DONE:", mode)
        print(summary)

    finally:
        # Crucial: no model / optimizer / loader / iterator survives.
        try:
            del df
        except Exception:
            pass
        del optimizer
        del loader
        del model
        cleanup_cuda()
        memory_status(f"after {mode} cleanup:")

    return summary


## 10. Run selected mode(s)

In [ ]:

all_summaries = []

for mode in RUN_MODES:
    summary = run_refinement(mode)
    all_summaries.append(summary)

summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(
    OUTPUT_ROOT / "summary.csv",
    index=False,
)

display(summary_df)

print("\nAll requested runs finished.")
print("Outputs:", OUTPUT_ROOT)
memory_status("final:")
